#### 4.1.1 Node-style hooks

此类钩子都通过**runtime.context**访问上下文对象。

用户自定义**ContextSchema**用`@dataclass`修饰，在Agent创建时通过**state_schema**参数传递。

本例从长期记忆中查询用户额度，如果额度用尽则中断流程。

In [1]:
from dataclasses import dataclass
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState,hook_config
from langchain.messages import AIMessage

from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from loguru import logger
from langchain_openai import ChatOpenAI

load_dotenv()

store = InMemoryStore()
namespace = ("users", "credits")
key1 = 'Ada Lovelace'
value1 = {
    "tokens_credit_left": 5000,
    "user_level": 5
}
key2 = 'Blackwell'
value2 = {
    "tokens_credit_left": 2999,
    "user_level": 5
}
key3 = 'Ampere'
value3 = {
    "tokens_credit_left": 1000,
    "user_level": 5
}
store.put(namespace, key1, value1)
store.put(namespace, key2, value2)
store.put(namespace, key3, value3)

@dataclass
class UserContext:
    username: str

class CheckCredit(AgentMiddleware):
    @hook_config(can_jump_to=["end"])
    def before_model(self, state: AgentState, runtime: Runtime) ->dict[str, Any] | None:
        store = runtime.store
        context = runtime.context

        username = context.username
        credit = store.get(namespace, username)
        if not credit:
            return {
                "jump_to": "end",
                "messages": AIMessage("您尚未注册~")
            }

        if credit.value["tokens_credit_left"] < 3000:
            return {
                "jump_to": "end",
                "messages": AIMessage(f"{username}额度不足，请充值")
            }

        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str,Any] | None:
        store = runtime.store
        context = runtime.context

        username = context.username
        credit = store.get(namespace, username)
        usage_metadata = state["messages"][-1].usage_metadata
        token_usage = usage_metadata["input_tokens"] + usage_metadata["output_tokens"] * 6
        credit.value["tokens_credit_left"] -= token_usage

        # 更新长期记忆
        store.put(namespace, username, credit.value)

        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str,Any] | None:
        store = runtime.store
        context = runtime.context

        username = context.username
        item = store.get(namespace, username)
        credit = item.value["tokens_credit_left"]

        logger.info(f"{username} 当前剩余额度：{credit}")
        return None

# 1.实例化模型
model = ChatOpenAI(
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

agent = create_agent(
    model=model,
    middleware=[CheckCredit()],
    store=store,
    context_schema=UserContext
)

response1 = agent.invoke(
    {"messages": ["你好啊，你知道 Ada Lovelace 的贡献吗？"]},
    context=UserContext(username="Ada Lovelace")
)
print("=" * 30, '-> Ada Lovelace messages <-', '=' * 30)
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent.invoke(
    {"messages": ["你好啊，你知道 Blackwell 的贡献吗？"]},
    context=UserContext(username="Blackwell")
)
print("=" * 30, '-> Blackwell messages <-', '=' * 30)
for msg in response2["messages"]:
    msg.pretty_print()

2026-11-03 19:43:19.045 | INFO     | __main__:after_agent:86 - Ada Lovelace 当前剩余额度：-2749
2026-11-03 19:43:19.130 | INFO     | __main__:after_agent:86 - Blackwell 当前剩余额度：2999


============================== -> Ada Lovelace messages <- ==============================
================================ Human Message =================================

你好啊，你知道 Ada Lovelace 的贡献吗？
================================== Ai Message ==================================

你好！很高兴为你解答。

**Ada Lovelace（爱达·洛夫莱斯）** 是科技史上极其重要的人物，她被广泛认为是**世界上第一位计算机程序员**。她的贡献不仅在于编写了第一段代码，更在于她对计算机本质的**超前预见**。

以下是她主要的贡献：

### 1. 编写了世界上第一个“计算机程序”
Ada Lovelace 与著名的发明家**查尔斯·巴贝奇（Charles Babbage）**合作研究“**分析机**”（Analytical Engine）。
*   **背景：** 分析机是巴贝奇设计的一种机械式通用计算机，虽然在当时未能完全建成，但它包含了现代计算机的核心概念（如处理器、内存、循环和条件分支）。
*   **贡献：** 在她翻译和注释巴贝奇关于分析机的论文时，她添加了大量的“注释”（Notes）。其中著名的“Note G”详细描述了如何利用分析机计算**伯努利数（Bernoulli numbers）**。这被认为是人类历史上第一段复杂的、可用于机器执行的算法。

### 2. 提出了“通用计算机”的宏伟愿景（最伟大的贡献）
这是 Ada Lovelace 最令人惊叹的贡献。当时的巴贝奇主要把他的机器看作是“算术工具”，即只能处理数字。但 Ada 却提出了更高维度的思考：
*   **超越计算：** 她意识到，如果机器能够处理符号（例如数字、字母、甚至音符），那么它就可以用来处理任何能够用规则表达的东西。
*   **多媒体雏形：** 她预言，如果机器能处理音乐或图形的逻辑，它甚至可以“创作音乐”或“生成图像”。
*   **核心逻辑：** 她意识到**机器不需要具备“智慧”也能执行复杂的指令**

#### 4.1.2 Wrap-style hooks

- **①wrap_model_call**

本例通过 **context**中记录的用户身份信息调整暴露给模型的工具集。

此处更改模型请求的方法是 **transient request update**，见上文**短期记忆2-4-2**

In [2]:
from dataclasses import dataclass
from typing import Callable
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelResponse,ModelRequest
from langchain.messages import AIMessage
from langchain.tools import tool

from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from loguru import logger

load_dotenv()

store = InMemoryStore()
namespace = ("users", "credits")
key1 = 'Ada Lovelace'
value1 = {
    "tokens_left": 3000,  # 剩余大模型token额度
    "get_weather": "yes",  # 是否认证天气查询服务
    "get_news": "no"  # 是否认证网络爬虫服务，用于查询新闻
}
key2 = 'Blackwell'
value2 = {
    "tokens_left": 2000,  # 剩余大模型token额度
    "get_weather": "no",  # 是否认证天气查询服务
    "get_news": "yes"  # 是否认证网络爬虫服务，用于查询新闻
}
key3 = 'Ampere'
value3 = {
    "tokens_left": 3000,  # 剩余大模型token额度
    "get_weather": "yes",  # 是否认证天气查询服务
    "get_news": "yes"  # 是否认证网络爬虫服务，用于查询新闻
}
store.put(namespace, key1, value1)
store.put(namespace, key2, value2)
store.put(namespace, key3, value3)

@tool
def get_weather(city: str):
    """
    查询指定城市当日天气

    Args:
        city: 城市名称
    """
    return f"{city} 今天天气不错"

@tool()
def get_news():
    """查询当日新闻"""
    return "美伊尚未达成停战协议"

@dataclass
class UserContext:
    username: str

class CheckCredit(AgentMiddleware):
    def wrap_model_call(
            self,
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        context = request.runtime.context
        store = request.runtime.store

        value = store.get(namespace, context.username).value
        tools = []

        for tool in request.tools:
            tool_name = tool.name
            if value[tool_name] == "yes":
                tools.append(tool)
            else:
                logger.warning(f"{context.username} 无权调用 {tool_name}")

        # 更改调用请求携带的工具集，仅本次生效
        request = request.override(tools=tools)

        return handler(request)

# 1.实例化模型
model = ChatOpenAI(
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

agent = create_agent(
    model=model,
    middleware=[CheckCredit()],
    tools=[get_weather, get_news],
    store=store,
    context_schema=UserContext
)

response1 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context=UserContext(username="Ada Lovelace")
)
print("=" * 30, '-> Ada Lovelace messages <-', '=' * 30)
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context=UserContext(username="Blackwell")
)
print("=" * 30, '-> Blackwell messages <-', '=' * 30)
for msg in response2["messages"]:
    msg.pretty_print()

response3 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context=UserContext(username="Ampere")
)
print("=" * 30, '-> Ampere messages <-', '=' * 30)
for msg in response3["messages"]:
    msg.pretty_print()

2026-11-03 19:51:34.801 | WARNING  | __main__:wrap_model_call:75 - Ada Lovelace 无权调用 get_news
2026-11-03 19:52:52.961 | WARNING  | __main__:wrap_model_call:75 - Ada Lovelace 无权调用 get_news
2026-11-03 19:53:11.527 | WARNING  | __main__:wrap_model_call:75 - Blackwell 无权调用 get_weather


============================== -> Ada Lovelace messages <- ==============================
================================ Human Message =================================

你好啊，今天北京天气如何，今日有哪些新闻？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_fhjf3st0)
 Call ID: call_fhjf3st0
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京 今天天气不错
================================== Ai Message ==================================

你好！今天北京的天气不错。

至于今日新闻，由于我目前无法直接获取实时的全球新闻资讯，建议您可以通过新闻客户端、搜索引擎或社交媒体查看最新的实时新闻动态。


2026-11-03 19:54:31.559 | WARNING  | __main__:wrap_model_call:75 - Blackwell 无权调用 get_weather


============================== -> Blackwell messages <- ==============================
================================ Human Message =================================

你好啊，今天北京天气如何，今日有哪些新闻？
================================== Ai Message ==================================
Tool Calls:
  get_news (call_t4w3gy6s)
 Call ID: call_t4w3gy6s
  Args:
================================= Tool Message =================================
Name: get_news

美伊尚未达成停战协议
================================== Ai Message ==================================

抱歉，我目前无法获取实时的天气信息，建议您查看手机自带的天气应用或通过搜索引擎查询。

关于今日新闻，目前的一条重要消息是：**美伊尚未达成停战协议**。
============================== -> Ampere messages <- ==============================
================================ Human Message =================================

你好啊，今天北京天气如何，今日有哪些新闻？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_zykexdi5)
 Call ID: call_zykexdi5
  Args:
    city: 北京
  get_news (call_r961knly)
 Call 

#### 4.1.3 特殊的便捷中间件dynamic_prompt

如前所述，@dynamic_prompt底层实现机制和通用钩子函数是统一的，对于动态提示词处理提供了更加便捷的入口。

本例通过静态上下文中携带的用户信息，动态更改系统提示词。

In [3]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_openai import ChatOpenAI
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from loguru import logger

load_dotenv()

# 准备历史记忆
namespace = ("users", "preferences")
key1 = "Ada Lovelace"
value1 = {
    "chat_preferences": [
        "不喜欢啰嗦",
        "尽可能用最精简的文字解释清楚"
    ]
}

key2 = "Blackwell"
value2 = {
    "chat_preferences": [
        "喜欢长篇大论、引经据典",
        "不知道的东西要明确说'不知道'，不能胡编乱造"
    ]
}

store = InMemoryStore()
store.put(namespace, key1, value1)
store.put(namespace, key2, value2)

@dataclass
class UserContext:
    username: str

@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    username = request.runtime.context.username
    store = request.runtime.store

    preferences = store.get(namespace, username).value["chat_preferences"]
    custom_prompt = f"# 用户偏好\n{'\n'.join(preferences)}"
    logger.info(f"{username} 自定义系统提示词\n{custom_prompt}")

    return custom_prompt
model = ChatOpenAI(
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

agent = create_agent(
    model=model,
    middleware=[personalized_prompt],
    store=store,
    context_schema=UserContext
)

response1 = agent.invoke(
    {"messages": ["为什么花儿这样红？"]},
    context=UserContext(username="Ada Lovelace")
)
print("=" * 30, '-> Ada Lovelace messages <-', '=' * 30)
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent.invoke(
    {"messages": ["为什么花儿这样红？"]},
    context=UserContext(username="Blackwell")
)
print("=" * 30, '-> Blackwell messages <-', '=' * 30)
for msg in response2["messages"]:
    msg.pretty_print()

2026-11-03 20:00:32.638 | INFO     | __main__:personalized_prompt:44 - Ada Lovelace 自定义系统提示词
# 用户偏好
不喜欢啰嗦
尽可能用最精简的文字解释清楚
2026-11-03 20:02:14.465 | INFO     | __main__:personalized_prompt:44 - Blackwell 自定义系统提示词
# 用户偏好
喜欢长篇大论、引经据典
不知道的东西要明确说'不知道'，不能胡编乱造


============================== -> Ada Lovelace messages <- ==============================
================================ Human Message =================================

为什么花儿这样红？
================================== Ai Message ==================================

1. **色素**：含有花青素等物质。
2. **功能**：吸引昆虫或鸟类来传粉。
============================== -> Blackwell messages <- ==============================
================================ Human Message =================================

为什么花儿这样红？
================================== Ai Message ==================================

这是一个看似简单、实则蕴含着生物化学、进化生物学、物理光学以及美学哲学深度的问题。要回答“为什么花儿这样红”，我们不能仅仅停留在表面色彩的描述上，而必须从微观的分子结构到宏观的进化博弈，从自然的生存法则到人类的情感投射，进行多维度的剖析。

我们可以从以下四个维度来深度解析这一现象：

### 一、 分子层面的化学基础：花色苷的魔法

从生物化学的角度来看，花朵呈现红色，主要是因为植物体内合成并积累了特定的色素。其中最核心的家族是**花色苷（Anthocyanins）**。

1.  **化学结构与光谱吸收：** 花色苷是一种水溶性的真空色素（vacuolar pigments），属于黄酮类化合物。它们之所以呈现红色，是因为其分子结构能够吸收特定波长的光（主要是绿光和黄光），而反射出较长波长的红光。这就像是自然界的“滤镜”，通过分子的排列组合，将光能转化为视觉上的色彩。
2.  **协同色素：** 除了花色苷，某些花卉中还含有**类胡萝卜素（Carote

### 4.2 工具中访问

在上文介绍工具的自定义args_schema时我们介绍了两种方式：

- 基于pydantic模型
- 基于json_schema

实际上，Langchain底层在生成工具的schema时，如果参数类型注解是pydantic模型，也会被一并解析。

In [4]:
from pydantic import BaseModel, Field
from typing import List
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.prebuilt.tool_node import ToolRuntime
from langchain_core.utils.function_calling import convert_to_openai_tool

class UserInfo(BaseModel):
    username: str = Field(description="用户名")
    age: int = Field(description="年龄")
    hobbies: List[str] = Field(description="兴趣爱好")

@tool(parse_docstring=True)
def save_user_info(user_info: UserInfo, runtime: ToolRuntime) -> str:
    """
    保存用户信息

    Args:
        user_info: 用户信息对象
    """
    return "ok"

print(convert_to_openai_tool(save_user_info))

{'type': 'function', 'function': {'name': 'save_user_info', 'description': '保存用户信息', 'parameters': {'properties': {'user_info': {'properties': {'username': {'description': '用户名', 'type': 'string'}, 'age': {'description': '年龄', 'type': 'integer'}, 'hobbies': {'description': '兴趣爱好', 'items': {'type': 'string'}, 'type': 'array'}}, 'required': ['username', 'age', 'hobbies'], 'type': 'object', 'description': '用户信息对象'}}, 'required': ['user_info'], 'type': 'object'}}}


下面的案例中，我们通过工具读写长期记忆中保存的用户信息

In [7]:
from pydantic import BaseModel, Field
from typing import List, Any
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage, SystemMessage

from langgraph.prebuilt.tool_node import ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver

from loguru import logger
from dotenv import load_dotenv
load_dotenv()

@dataclass
class UserContext:
    user_id: str

store = InMemoryStore()
namespace = ("users", "user_info")

class UserInfo(BaseModel):
    username: str = Field(description="用户名", default="unknown")
    age: int = Field(description="年龄", default=0)
    # 这里相当于告诉模型，每次创建新实例时，调用list()创建一个新的list传进去
    hobbies: List[str] = Field(description="兴趣爱好", default_factory=list)
    # 这种写法是用可变对象做默认值，会有多个对象共享一个列表对象的风险
    # hobbies: List[str] = Field(description="兴趣爱好", default=[])

@tool(parse_docstring=True)
def read_user_info(runtime: ToolRuntime[UserContext, Any]) -> UserInfo |str:
    """
    读取用户信息

    Returns:
        UserInfo | str: 如果找到用户信息，返回 UserInfo 对象；否则返回空字符串。
    """
    user_id = runtime.context.user_id
    store = runtime.store
    item = store.get(namespace, user_id)
    if item:
        return UserInfo(**item.value)
    return ''

@tool(parse_docstring=True)
def write_user_info(user_info: UserInfo, runtime: ToolRuntime[UserContext,Any]) -> bool:
    """
    将用户信息写入长期记忆

    Args:
        user_info：用户信息对象

    Returns:
        bool: 如果成功写入则返回True，否则返回False
    """
    user_id = runtime.context.user_id
    store = runtime.store
    try:
        logger.info(f"写入记忆, namespace: {namespace}, user_id: {user_id},user_info: {user_info.model_dump()}")
        store.put(namespace, user_id, user_info.model_dump())
    except Exception as e:
        logger.error(e)
        return False
    return True

model = ChatOpenAI(
    model="gemma4:12b",
    api_key="sk_1234",
    base_url="http://localhost:11434/v1"
)

agent = create_agent(
    model =model,
    tools = [read_user_info, write_user_info],
    store = store,
    checkpointer=InMemorySaver(),
    context_schema=UserContext
)

config1 = {"configurable": {"thread_id": "thread_1"}}
config2 = {"configurable": {"thread_id": "thread_2"}}

agent.invoke(
    {"messages": [
        SystemMessage("如果输入包含用户信息，抽取并调用工具记录。在记录之前先查找历史信息，和新增信息合并后记录。"
                      "如果用户提问设计个人信息，调用工具查找"),
        HumanMessage("你好，我是韩立，我喜欢修仙")
    ]},
    context = UserContext(user_id = 'user_1'),
    config = config1
)

thread_1_response = agent.invoke(
    {"messages": ["我今年两百岁了，是你们口中的 “元婴老怪”，我喜欢跑步"]},
    context = UserContext(user_id = 'user_1'),
    config = config1
)

print("=" * 30, '-> thread_1 messages <-', '=' * 30)
for msg in thread_1_response["messages"]:
    msg.pretty_print()

thread_2_response = agent.invoke(
    {"messages": ["你还记得我吗？"]},
    context = UserContext(user_id = 'user_1'),
    config = config2
)

print("=" * 30, '-> thread_2 messages <-', '=' * 30)
for msg in thread_2_response["messages"]:
    msg.pretty_print()

2026-11-03 20:31:32.693 | INFO     | __main__:write_user_info:61 - 写入记忆, namespace: ('users', 'user_info'), user_id: user_1,user_info: {'username': '韩立', 'age': 0, 'hobbies': ['修仙']}
2026-11-03 20:36:16.435 | INFO     | __main__:write_user_info:61 - 写入记忆, namespace: ('users', 'user_info'), user_id: user_1,user_info: {'username': '韩立', 'age': 200, 'hobbies': ['修仙', '跑步']}


============================== -> thread_1 messages <- ==============================
================================ System Message ================================

如果输入包含用户信息，抽取并调用工具记录。在记录之前先查找历史信息，和新增信息合并后记录。如果用户提问设计个人信息，调用工具查找
================================ Human Message =================================

你好，我是韩立，我喜欢修仙
================================== Ai Message ==================================
Tool Calls:
  read_user_info (call_m6o3zgyh)
 Call ID: call_m6o3zgyh
  Args:
================================= Tool Message =================================
Name: read_user_info


================================== Ai Message ==================================
Tool Calls:
  write_user_info (call_8isvqit5)
 Call ID: call_8isvqit5
  Args:
    user_info: {'hobbies': ['修仙'], 'username': '韩立'}
================================= Tool Message =================================
Name: write_user_info

true
================================== Ai Message ==================================

你好，韩立！